# Course 5: Winning Online AI Competitions
## Session 11: Reasoning competitions: sample and vote, with and without a tool

**Goal:** measure what voting over several samples adds, and what changes when the solver may run Python.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026) on synthetic or built-in data. Cells labelled NOT RUN need a Kaggle account, a competition's data, a GPU or an extra library.

### 1. The setting
In competitions such as the AI Mathematical Olympiad, a model must return one integer per problem, offline and within a time limit. Strong solutions sample many attempts and vote, and let the model write and run code. This notebook uses a simulated solver so the statistics can be studied on a CPU; the real model goes in the last cell.

### 2. Problems and two simulated solvers
Problems are arithmetic expressions with a known answer.
- The **direct** solver 'does the arithmetic in its head': each operation has a chance of a slip, and slips produce scattered wrong answers.
- The **tool** solver writes a Python expression and runs it. The arithmetic is then exact, but it sometimes writes the wrong program (a misread operator), and that mistake tends to repeat.

In [ ]:
import random, re, numpy as np
from collections import Counter
def make_problem(rng, n_ops):
    nums = [rng.randint(2, 30) for _ in range(n_ops + 1)]; ops = [rng.choice(["+", "-", "*"]) for _ in range(n_ops)]
    expr = str(nums[0]) + "".join(f" {o} {v}" for o, v in zip(ops, nums[1:]))
    return expr, eval(expr)
rng = random.Random(0)
problems = [make_problem(rng, n_ops) for n_ops in [2, 3, 4, 5, 6, 8] for _ in range(50)]

def direct_solver(expr, rng, slip=0.12):
    """Evaluates left to right with correct precedence, but each operation may go slightly wrong."""
    tokens = expr.split(); terms, sign = [], 1; cur = int(tokens[0])
    for op, v in zip(tokens[1::2], tokens[2::2]):
        v = int(v)
        if op == "*": cur = cur * v
        else: terms.append(sign * cur); sign = 1 if op == "+" else -1; cur = v
        if rng.random() < slip: cur += rng.choice([-10, -2, -1, 1, 2, 10])              # an arithmetic slip
    return sum(terms) + sign * cur

def tool_solver(expr, rng, misread=0.15):
    """Writes a program. With some probability it misreads one operator; the same misreading is likely on every attempt."""
    program = expr
    h = random.Random(hash(expr) % 10 ** 6)                                             # the problem's 'trap', fixed per problem
    trap_pos = h.randrange(len(expr.split()) // 2); trap_prone = h.random() < 0.5
    if rng.random() < (misread * 3 if trap_prone else misread / 3):
        toks = expr.split(); i = 1 + 2 * trap_pos; toks[i] = {"+": "-", "-": "+", "*": "+"}[toks[i]]; program = " ".join(toks)
    return program

### 3. Running generated code safely
Never `eval` a model's output directly. At minimum: allow only what you expect, run with a time limit, and catch every error. In a real system, run the code in a separate process with no network and limited memory.

In [ ]:
import subprocess, sys
def run_python(code, timeout=2.0):
    """Run code in a separate Python process and return its last printed line, or None on any failure."""
    try:
        r = subprocess.run([sys.executable, "-I", "-c", code], capture_output=True, text=True, timeout=timeout)
        return r.stdout.strip().splitlines()[-1] if r.returncode == 0 and r.stdout.strip() else None
    except subprocess.TimeoutExpired:
        return None
print("normal program :", run_python("print(17 * 3 - 4)"))
print("infinite loop  :", run_python("while True: pass", timeout=0.5))
print("crash          :", run_python("print(1 / 0)"))

def safe_arithmetic(expr):
    """For this notebook's expressions a whitelist is enough and much faster than a process per call."""
    return eval(expr) if re.fullmatch(r"[0-9+\-* ]+", expr) else None
assert run_python("print(17 * 3 - 4)") == "47" and run_python("while True: pass", timeout=0.5) is None and safe_arithmetic("__import__('os')") is None

### 4. Sample and vote
Draw k attempts and return the most common answer.

In [ ]:
def vote(answers):
    answers = [a for a in answers if a is not None]
    return Counter(answers).most_common(1)[0][0] if answers else None

def accuracy(solver_kind, k, trials=3, seed=0):
    rng = random.Random(seed); correct = 0
    for _ in range(trials):
        for expr, truth in problems:
            if solver_kind == "direct": answers = [direct_solver(expr, rng) for _ in range(k)]
            else: answers = [safe_arithmetic(tool_solver(expr, rng)) for _ in range(k)]
            correct += vote(answers) == truth
    return correct / (trials * len(problems))

print("samples   direct solver   solver with Python")
table = {}
for k in [1, 3, 5, 9, 17, 33]:
    table[k] = (accuracy("direct", k), accuracy("tool", k)); print(f"{k:7d}   {table[k][0]:13.3f}   {table[k][1]:18.3f}")
assert table[33][0] > table[1][0] + 0.1 and table[1][1] > table[1][0]

Two different curves.
- The direct solver starts low and climbs steeply: its errors are scattered, so the correct answer is usually the single most common one even when it is rare.
- The tool solver starts high and gains less: when it misreads a problem, it tends to misread it the same way every time, and no amount of voting fixes a shared mistake.

Voting removes independent errors. It cannot remove systematic ones. To fix those you need diversity: different prompts, different approaches, or a check.

### 5. By difficulty

In [ ]:
rng = random.Random(1)
print("operations   direct k=1   direct k=17   tool k=1   tool k=17")
for i, n_ops in enumerate([2, 3, 4, 5, 6, 8]):
    sub = problems[i * 50:(i + 1) * 50]
    def acc(kind, k): return np.mean([vote([direct_solver(e, rng) if kind == "d" else safe_arithmetic(tool_solver(e, rng)) for _ in range(k)]) == t for e, t in sub])
    print(f"{n_ops:10d}   {acc('d', 1):10.2f}   {acc('d', 17):11.2f}   {acc('t', 1):8.2f}   {acc('t', 17):9.2f}")

### 6. Combining both, and stopping early
Run both kinds of attempt and vote over all of them: the two solvers fail in different ways. And stop sampling once the leader cannot be caught, which saves time for the hard problems.

In [ ]:
def mixed_vote_with_early_stop(expr, rng, max_k=16, lead=4):
    answers = []
    for i in range(max_k):
        answers.append(direct_solver(expr, rng) if i % 2 else safe_arithmetic(tool_solver(expr, rng)))
        top = Counter(answers).most_common(2)
        if top[0][1] - (top[1][1] if len(top) > 1 else 0) >= lead: break                  # the leader is far enough ahead
    return vote(answers), len(answers)
rng = random.Random(2); out = [mixed_vote_with_early_stop(e, rng) for e, _ in problems]
acc_mixed = np.mean([a == t for (a, _), (_, t) in zip(out, problems)]); mean_k = np.mean([k for _, k in out])
print(f"mixed voting with early stopping: accuracy {acc_mixed:.3f} using {mean_k:.1f} samples per problem on average (limit 16)")
print(f"for comparison at 17 samples: direct {table[17][0]:.3f} | tool {table[17][1]:.3f}")

### 7. A time budget across problems
With 50 problems and a fixed total time, do not give every problem the same share. Give each a base allowance, stop early on easy ones, and pass the saved time to the rest. And always return an answer: an integer guess scores more than a timeout.

### 8. With a real model (NOT RUN here)

In [ ]:
# NOT RUN: needs GPUs and an open model that fits the competition's hardware; in competition, everything is loaded offline.
PROMPT_DIRECT = "Solve the problem. Think step by step, then give the final answer as an integer inside \\boxed{{}}.\n\n{problem}"
PROMPT_TOOL = "Solve the problem by writing Python code inside a ```python block. The code must print only the final integer.\n\n{problem}"

def extract_boxed(text):
    m = re.findall(r"\\boxed\{(-?\d+)\}", text); return int(m[-1]) if m else None
def extract_code(text):
    m = re.findall(r"```python\n(.*?)```", text, flags=re.S); return m[-1] if m else None

# def solve(problem, llm, k=16):
#     answers = []
#     for i in range(k):
#         if i % 2 == 0:
#             out = run_python(extract_code(llm(PROMPT_TOOL.format(problem=problem), temperature=0.8)) or "", timeout=10)
#             answers.append(int(out) if out and out.lstrip("-").isdigit() else None)
#         else:
#             answers.append(extract_boxed(llm(PROMPT_DIRECT.format(problem=problem), temperature=0.8)))
#     return vote(answers) or 0
# Evaluate on 50 problems you did not tune on. Report accuracy at k = 1, 4, 16 for each mode, and the time per problem.

### TODO
1. Make the tool solver's mistakes independent across attempts (remove the per-problem trap). How does its voting curve change?
2. Add a verifier: an answer counts only if two different methods agree. What happens to accuracy and to the number of unanswered problems?
3. Run the real-model cell on 20 problems from a past public set and compare the two modes.